# H3 Feature Engineering - Silver Layer

Builds H3 resolution-8 feature base from Census demographics and OSM POI data.
Replaces the CARTO Marketplace dependency with first-party feature engineering.

**Algorithm:**
1. Generate H3 res-8 grid covering all training states
2. Area-weighted demographic aggregation from Census block groups to H3 cells
3. POI count aggregation using H3 indexing (fast point-to-cell assignment)
4. Derive features: `urbanity`, `human_activity_index`, `target_demographic_total`,
   `pct_college_educated`, `unemployment_rate`, `total_poi_count_norm`

**Inputs:**
- `{bronze}.census_blockgroups` — Block group geometries (all training states)
- `{bronze}.census_demographics` — ACS demographic data (all training states)
- `{bronze}.osm_pois_raw` — General POI categories (all training states)
- `{bronze}.census_states` — State boundaries (for H3 grid generation)

**Output:**
- `{silver}.h3_features_clean` — H3-level features for trade area enrichment

## Parameters

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit, coalesce, explode, when
from pyspark.sql.window import Window

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("state_filter", "MA,MI,VA,NY,WA,MD,NJ")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
state_filter = dbutils.widgets.get("state_filter")

assert catalog and bronze_schema and silver_schema, "Missing required parameters"

state_list = [s.strip() for s in state_filter.split(",") if s.strip()]

# Table references
bg_table = f"{catalog}.{bronze_schema}.census_blockgroups"
demo_table = f"{catalog}.{bronze_schema}.census_demographics"
pois_table = f"{catalog}.{bronze_schema}.osm_pois_raw"
states_table = f"{catalog}.{bronze_schema}.census_states"
output_table = f"{catalog}.{silver_schema}.h3_features_clean"

print(f"States: {state_list}")
print(f"Output: {output_table}")

## Step 1: Generate H3 Grid for Training States

Create H3 res-8 cells covering all target states. Uses hierarchical approach:
cover at res 5 (coarse), then explode to res 8 children.

In [ ]:
# Generate H3 res-8 grid for all training states
state_boundaries = (
    spark.table(states_table)
    .filter(col("state_abbr").isin(state_list))
    .select("state_abbr", "geometry")
)

print(f"Generating H3 grid for {state_boundaries.count()} states...")

h3_grid = (
    state_boundaries
    .select(
        "state_abbr",
        explode(expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3")
    )
    .select(
        "state_abbr",
        explode(expr("h3_tochildren(coarse_h3, 8)")).alias("h3_cell_id")
    )
    .distinct()
    .withColumn("h3_geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)"))
)

h3_count = h3_grid.count()

print(f"H3 grid: {h3_count:,} cells at resolution 8")
print("\nCells by state:")
display(h3_grid.groupBy("state_abbr").count().orderBy("state_abbr"))

## Step 2: Area-Weighted Demographic Aggregation

For each H3 cell, intersect with Census block groups and apportion demographic
counts by the ratio of intersection area to total block group area.

In [ ]:
# Load block groups and demographics
block_groups = (
    spark.table(bg_table)
    .filter(col("state_abbr").isin(state_list))
    .select("geoid", "state_abbr", "geometry")
)

demographics = spark.table(demo_table)

# Build geoid from demographics table components
demographics = demographics.withColumn(
    "geoid", F.concat(col("state"), col("county"), col("tract"), col("block_group"))
)

print(f"Block groups: {block_groups.count():,}")
print(f"Demographics records: {demographics.count():,}")

In [ ]:
# Spatial intersection: H3 cells x block groups
# For each pair, compute the fraction of the block group that falls in the H3 cell

h3_bg_intersect = (
    h3_grid.alias("h3")
    .join(
        block_groups.alias("bg"),
        (col("h3.state_abbr") == col("bg.state_abbr")) &
        expr("ST_Intersects(h3.h3_geometry, bg.geometry)"),
        "inner"
    )
    .select(
        col("h3.h3_cell_id"),
        col("h3.state_abbr"),
        col("bg.geoid"),
        # Area-weight ratio: intersection area / block group area
        (expr("ST_Area(ST_Intersection(h3.h3_geometry, bg.geometry))") /
         expr("ST_Area(bg.geometry)")).alias("intersection_ratio")
    )
    # Filter out negligible overlaps (< 0.1% of block group area)
    .filter(col("intersection_ratio") > 0.001)
)

print(f"H3-BlockGroup intersection pairs: {h3_bg_intersect.count():,}")

In [ ]:
# Join intersections with demographics and apply area weighting
# Separate count variables (area-weighted SUM) from rate/median variables (area-weighted AVG)

# Count-based variables: area-weighted sum apportions counts to H3 cells
count_cols = [
    "total_population",
    "total_households",
    # Target demographic bins (Census splits 20-24 into 20, 21, 22-24)
    "male_20", "male_21", "male_22_to_24",
    "male_25_to_29", "male_30_to_34",
    "female_20", "female_21", "female_22_to_24",
    "female_25_to_29", "female_30_to_34",
    # Education counts (for deriving pct_college_educated)
    "high_school_grad", "bachelors_degree", "masters_degree",
    "professional_degree", "doctorate_degree",
    # Employment counts (for deriving unemployment_rate)
    "in_labor_force", "employed", "unemployment_count",
    # Housing counts
    "vacant_housing_units",
]

# Rate/median variables: area-weighted average (sum(val*w)/sum(w))
median_cols = [
    "median_household_income",
    "per_capita_income",
    "median_home_value",
    "median_gross_rent",
]

all_demo_cols = count_cols + median_cols

# Filter to available columns
available_count_cols = [c for c in count_cols if c in demographics.columns]
available_median_cols = [c for c in median_cols if c in demographics.columns]
missing = [c for c in all_demo_cols if c not in demographics.columns]
if missing:
    print(f"WARNING: Missing demographic columns (will be 0): {missing}")

all_available = available_count_cols + available_median_cols

weighted = (
    h3_bg_intersect.alias("i")
    .join(demographics.alias("d"), col("i.geoid") == col("d.geoid"), "inner")
    .select(
        col("i.h3_cell_id"),
        col("i.state_abbr"),
        col("i.intersection_ratio"),
        *[coalesce(col(f"d.{c}").cast("double"), lit(0.0)).alias(c) for c in all_available]
    )
)

# Count variables: sum(value * ratio) — apportions counts proportionally
count_agg_exprs = [
    F.sum(col(c) * col("intersection_ratio")).cast("long").alias(c)
    for c in available_count_cols
]

# Median/rate variables: sum(value * ratio) / sum(ratio) — proper weighted average
median_agg_exprs = [
    (F.sum(col(c) * col("intersection_ratio")) / F.sum(col("intersection_ratio"))).alias(c)
    for c in available_median_cols
]

h3_demographics = (
    weighted
    .groupBy("h3_cell_id", "state_abbr")
    .agg(*(count_agg_exprs + median_agg_exprs))
)

# Rename total_population -> population for output contract
h3_demographics = h3_demographics.withColumnRenamed("total_population", "population")

print(f"H3 cells with demographics: {h3_demographics.count():,}")
print(f"Count variables ({len(available_count_cols)}): {available_count_cols}")
print(f"Median variables ({len(available_median_cols)}): {available_median_cols}")
print("\nPopulation summary:")
display(h3_demographics.select("population").summary())

## Step 3: POI Count Aggregation

Use H3 indexing to assign each POI to its H3 cell (fast, no geometry join needed),
then pivot to get one column per category.

In [ ]:
# Assign each POI to its H3 cell using lat/lon → H3 index (no geometry join needed)
poi_h3 = (
    spark.table(pois_table)
    .filter(col("state").isin(state_list))
    .withColumn("h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)"))
    .groupBy("h3_cell_id", "poi_category")
    .agg(F.count("*").alias("poi_count"))
)

# Pivot to get one column per category
poi_categories = ["retail", "food_drink", "leisure", "education",
                  "healthcare", "financial", "tourism", "transportation"]

h3_pois = (
    poi_h3
    .groupBy("h3_cell_id")
    .pivot("poi_category", poi_categories)
    .agg(F.sum("poi_count"))
)

# Fill nulls with 0 for missing categories
for cat in poi_categories:
    h3_pois = h3_pois.withColumn(cat, coalesce(col(cat), lit(0)).cast("long"))

print(f"H3 cells with POI data: {h3_pois.count():,}")
print("\nPOI category totals:")
display(h3_pois.select([F.sum(c).alias(c) for c in poi_categories]))

## Step 4: Join Demographics + POIs and Derive Features

In [ ]:
# Join demographics with POI counts on h3_cell_id
h3_features = (
    h3_demographics.alias("demo")
    .join(h3_pois.alias("poi"), "h3_cell_id", "left")
)

# Fill missing POI columns with 0
for cat in poi_categories:
    h3_features = h3_features.withColumn(cat, coalesce(col(cat), lit(0)))

# --- Combine Census sub-bins into age ranges ---
# male_20_to_24 = male_20 + male_21 + male_22_to_24
h3_features = h3_features.withColumn(
    "male_20_to_24",
    coalesce(col("male_20"), lit(0)) +
    coalesce(col("male_21"), lit(0)) +
    coalesce(col("male_22_to_24"), lit(0))
).withColumn(
    "female_20_to_24",
    coalesce(col("female_20"), lit(0)) +
    coalesce(col("female_21"), lit(0)) +
    coalesce(col("female_22_to_24"), lit(0))
)

# --- Derived features ---

# target_demographic_total: sum of 20-34 age bins
h3_features = h3_features.withColumn(
    "target_demographic_total",
    coalesce(col("male_20_to_24"), lit(0)) + coalesce(col("female_20_to_24"), lit(0)) +
    coalesce(col("male_25_to_29"), lit(0)) + coalesce(col("female_25_to_29"), lit(0)) +
    coalesce(col("male_30_to_34"), lit(0)) + coalesce(col("female_30_to_34"), lit(0))
)

# total_poi_count: sum of all 8 categories
h3_features = h3_features.withColumn(
    "total_poi_count",
    sum([coalesce(col(c), lit(0)) for c in poi_categories])
)

# pct_college_educated: (bachelors + masters + professional + doctorate) / population
college_sum = (
    coalesce(col("bachelors_degree"), lit(0)) +
    coalesce(col("masters_degree"), lit(0)) +
    coalesce(col("professional_degree"), lit(0)) +
    coalesce(col("doctorate_degree"), lit(0))
)
h3_features = h3_features.withColumn(
    "pct_college_educated",
    when(col("population") > 0, F.round(college_sum / col("population"), 4))
    .otherwise(0.0)
)

# unemployment_rate: unemployment_count / in_labor_force
h3_features = h3_features.withColumn(
    "unemployment_rate",
    when(col("in_labor_force") > 0,
         F.round(coalesce(col("unemployment_count"), lit(0)) / col("in_labor_force"), 4))
    .otherwise(0.0)
)

print(f"Joined features: {h3_features.count():,} H3 cells")
print(f"\nDerived features added: target_demographic_total, total_poi_count, pct_college_educated, unemployment_rate")

In [ ]:
# --- Urbanity classification ---
# Decile-based on total POI count (deterministic replacement for CARTO's proprietary urbanity)

h3_features = h3_features.withColumn(
    "urbanity_decile",
    F.ntile(10).over(Window.orderBy("total_poi_count"))
).withColumn(
    "urbanity",
    when(col("urbanity_decile") >= 9, "Very_High_density_urban")
    .when(col("urbanity_decile") >= 7, "High_density_urban")
    .when(col("urbanity_decile") >= 5, "Medium_density_urban")
    .when(col("urbanity_decile") >= 3, "Low_density_urban")
    .otherwise("rural")
).withColumn(
    "urbanity_category",
    when(col("urbanity").isin("Very_High_density_urban", "High_density_urban"), "urban")
    .when(col("urbanity").isin("Medium_density_urban", "Low_density_urban"), "suburban")
    .otherwise("rural")
)

# --- Normalized POI count (min-max normalization) ---
poi_stats = h3_features.agg(
    F.min("total_poi_count").alias("min_poi"),
    F.max("total_poi_count").alias("max_poi")
).collect()[0]

poi_range = poi_stats["max_poi"] - poi_stats["min_poi"]
h3_features = h3_features.withColumn(
    "total_poi_count_norm",
    when(lit(poi_range) > 0,
         F.round((col("total_poi_count") - lit(poi_stats["min_poi"])) / lit(poi_range), 4))
    .otherwise(lit(0.0))
)

# --- Human activity index ---
# Composite score from population density + POI density, normalized to 0-100
w = Window.partitionBy()

pop_min = F.min("population").over(w)
pop_max = F.max("population").over(w)
poi_min = F.min("total_poi_count").over(w)
poi_max = F.max("total_poi_count").over(w)

h3_features = h3_features.withColumn(
    "pop_norm",
    when(pop_max == pop_min, lit(0.5))
    .otherwise((col("population") - pop_min) / (pop_max - pop_min))
).withColumn(
    "poi_norm",
    when(poi_max == poi_min, lit(0.5))
    .otherwise((col("total_poi_count") - poi_min) / (poi_max - poi_min))
).withColumn(
    "human_activity_index",
    F.round((lit(0.5) * col("pop_norm") + lit(0.5) * col("poi_norm")) * 100, 2)
)

print("Urbanity distribution:")
display(h3_features.groupBy("urbanity_category").count().orderBy("urbanity_category"))

print("\nHuman activity index summary:")
display(h3_features.select("human_activity_index").summary())

print(f"\nTotal POI count norm range: {poi_stats['min_poi']} - {poi_stats['max_poi']}")

## Step 5: Select Output Columns and Write

In [ ]:
# Select output columns for h3_features_clean
output_df = (
    h3_features
    .select(
        "h3_cell_id",
        "state_abbr",
        # Demographics (counts)
        col("population").cast("long"),
        col("total_households").cast("long"),
        col("male_20_to_24").cast("long"),
        col("female_20_to_24").cast("long"),
        col("male_25_to_29").cast("long"),
        col("female_25_to_29").cast("long"),
        col("male_30_to_34").cast("long"),
        col("female_30_to_34").cast("long"),
        col("target_demographic_total").cast("long"),
        # Demographics (rates/medians)
        col("median_household_income").cast("double"),
        col("per_capita_income").cast("double"),
        col("median_home_value").cast("double"),
        col("median_gross_rent").cast("double"),
        # Derived rates
        col("pct_college_educated").cast("double"),
        col("unemployment_rate").cast("double"),
        # POI counts
        col("retail").cast("long"),
        col("food_drink").cast("long"),
        col("leisure").cast("long"),
        col("education").cast("long"),
        col("healthcare").cast("long"),
        col("financial").cast("long"),
        col("tourism").cast("long"),
        col("transportation").cast("long"),
        col("total_poi_count").cast("long"),
        col("total_poi_count_norm").cast("double"),
        # Activity and urbanity
        "urbanity",
        col("human_activity_index").cast("double"),
        "urbanity_category",
    )
    .withColumn("processing_timestamp", F.current_timestamp())
)

# Write to silver
(
    output_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"Written {output_df.count():,} H3 features to {output_table}")
print(f"\nNew features added: median_household_income, per_capita_income, median_home_value,")
print(f"  median_gross_rent, pct_college_educated, unemployment_rate, total_poi_count_norm")

## Validation

In [ ]:
print("=" * 80)
print("H3 FEATURES VALIDATION")
print("=" * 80)

result = spark.table(output_table)

print(f"\nTotal H3 cells: {result.count():,}")

print("\nBy state:")
display(result.groupBy("state_abbr").agg(
    F.count("*").alias("cells"),
    F.avg("population").alias("avg_population"),
    F.avg("total_poi_count").alias("avg_poi_count"),
    F.avg("human_activity_index").alias("avg_activity_index"),
    F.avg("median_household_income").alias("avg_median_income"),
    F.avg("per_capita_income").alias("avg_per_capita_income"),
    F.avg("pct_college_educated").alias("avg_pct_college"),
    F.avg("unemployment_rate").alias("avg_unemployment"),
).orderBy("state_abbr"))

print("\nBy urbanity:")
display(result.groupBy("urbanity_category").agg(
    F.count("*").alias("cells"),
    F.avg("population").alias("avg_population"),
    F.avg("total_poi_count").alias("avg_poi_count"),
    F.avg("median_household_income").alias("avg_median_income"),
    F.avg("pct_college_educated").alias("avg_pct_college"),
).orderBy("urbanity_category"))

print("\nFeature distributions:")
display(result.select(
    "population", "target_demographic_total", "total_poi_count",
    "human_activity_index", "median_household_income", "per_capita_income",
    "pct_college_educated", "unemployment_rate", "median_home_value",
    "total_poi_count_norm"
).summary())

# Schema check
print("\nOutput schema:")
result.printSchema()

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)